In [3]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.1-cp311-cp311-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.1-cp311-cp311-win_amd64.whl (8.3 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
   ---------------------------------------- 0.0/490.6 kB ? eta -:--:--
   -- ------------------------------------- 30.7/490.6 kB 1.4 MB/s eta 0:00:01
   --- ----------------------------------- 41.0/490.6 kB 495.5 kB/s eta 0:00:01
   ------- ------------------------------- 92.2/490.6 kB 655.4 kB/s eta 0:00:01
   --------------- ------------------------ 194.6/490.6 kB 1.1 MB/s eta 0:00:01
   ---------------------------------------  481.3/490.6 kB 2.2 MB/s eta 0:00:01
   ---------------------------------------- 490.6/490.6 kB 2.1


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from pathlib import Path

# Load cleaned demand data
df = pd.read_csv(
    "../data/processed/demand_clean.csv",
    parse_dates=["Date"]
)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nDate range:")
print(df["Date"].min(), "to", df["Date"].max())

print("\nMissing values:")
print(df.isnull().sum())

display(df.head())

Dataset shape: (73100, 18)

Columns:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality', 'flag_negative_forecast', 'Demand Forecast Clean', 'Inventory Coverage']

Date range:
2022-01-01 00:00:00 to 2024-01-01 00:00:00

Missing values:
Date                        0
Store ID                    0
Product ID                  0
Category                    0
Region                      0
Inventory Level             0
Units Sold                  0
Units Ordered               0
Demand Forecast             0
Price                       0
Discount                    0
Weather Condition           0
Holiday/Promotion           0
Competitor Pricing          0
Seasonality                 0
flag_negative_forecast      0
Demand Forecast Clean       0
Inventory Coverage        673
dtype: int64


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,flag_negative_forecast,Demand Forecast Clean,Inventory Coverage
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,False,135.47,1.705175
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,False,144.04,1.416273
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,False,74.02,1.378006
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,False,62.18,7.542618
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,False,9.26,17.926566


In [2]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Sort chronologically
df = df.sort_values("Date").copy()

# Calendar features
df["day_of_week"] = df["Date"].dt.dayofweek
df["month"] = df["Date"].dt.month
df["day_of_year"] = df["Date"].dt.dayofyear

# Encode product and store identifiers
df["Store_Code"] = df["Store ID"].str.extract(r"(\d+)").astype(int)
df["Product_Code"] = df["Product ID"].str.extract(r"(\d+)").astype(int)

features = [
    "Store_Code",
    "Product_Code",
    "day_of_week",
    "month",
    "day_of_year",
    "Price",
    "Discount",
    "Competitor Pricing",
    "Holiday/Promotion"
]

X = df[features]
y = df["Units Sold"]

# Chronological 80/20 split
split_index = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# Train model
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MODEL: Random Forest Regressor")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")

C:\Users\likitha\AppData\Local\Temp\ipykernel_28812\3508318597.py:9: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["day_of_week"] = df["Date"].dt.dayofweek
C:\Users\likitha\AppData\Local\Temp\ipykernel_28812\3508318597.py:11: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  df["day_of_year"] = df["Date"].dt.dayofyear


MODEL: Random Forest Regressor
MAE:  88.96
RMSE: 108.38
R²:   -0.0027


In [3]:
# Sort by product, store and date
df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).copy()

# Historical sales features
grouped = df.groupby(["Store ID", "Product ID"])["Units Sold"]

df["sales_lag_1"] = grouped.shift(1)
df["sales_lag_7"] = grouped.shift(7)
df["sales_rolling_7"] = (
    df.groupby(["Store ID", "Product ID"])["Units Sold"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

# Remove rows without sufficient historical observations
model_df = df.dropna(
    subset=["sales_lag_1", "sales_lag_7", "sales_rolling_7"]
).copy()

features_improved = features + [
    "sales_lag_1",
    "sales_lag_7",
    "sales_rolling_7"
]

# Chronological split
split_index = int(len(model_df) * 0.8)

X_train = model_df[features_improved].iloc[:split_index]
X_test = model_df[features_improved].iloc[split_index:]

y_train = model_df["Units Sold"].iloc[:split_index]
y_test = model_df["Units Sold"].iloc[split_index:]

improved_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

improved_model.fit(X_train, y_train)

improved_predictions = improved_model.predict(X_test)

print("IMPROVED RANDOM FOREST")
print("MAE:", round(mean_absolute_error(y_test, improved_predictions), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, improved_predictions)), 2))
print("R²:", round(r2_score(y_test, improved_predictions), 4))

IMPROVED RANDOM FOREST
MAE: 89.61
RMSE: 109.8
R²: -0.0042


In [4]:
print("UNITS SOLD DISTRIBUTION")
display(df["Units Sold"].describe())

print("\nAVERAGE SALES BY DAY")
display(
    df.groupby("Date")["Units Sold"]
    .mean()
    .describe()
)

print("\nAVERAGE SALES BY PRODUCT")
display(
    df.groupby("Product ID")["Units Sold"]
    .mean()
    .describe()
)

UNITS SOLD DISTRIBUTION


count    73100.000000
mean       136.464870
std        108.919406
min          0.000000
25%         49.000000
50%        107.000000
75%        203.000000
max        499.000000
Name: Units Sold, dtype: float64


AVERAGE SALES BY DAY


count    731.00000
mean     136.46487
std       10.31959
min      106.42000
25%      129.81000
50%      136.73000
75%      143.23500
max      172.39000
Name: Units Sold, dtype: float64


AVERAGE SALES BY PRODUCT


count     20.000000
mean     136.464870
std        1.695694
min      133.468399
25%      135.416005
50%      136.440766
75%      137.476471
max      139.116826
Name: Units Sold, dtype: float64

In [5]:
# Inventory planning recommendations

planning = df.groupby(
    ["Store ID", "Product ID", "Category"],
    as_index=False
).agg(
    avg_daily_sales=("Units Sold", "mean"),
    avg_inventory=("Inventory Level", "mean"),
    avg_forecast=("Demand Forecast Clean", "mean")
)

# Illustrative 7-day coverage target
planning["target_stock_7_days"] = (
    planning["avg_forecast"] * 7
)

planning["reorder_gap"] = (
    planning["target_stock_7_days"] - planning["avg_inventory"]
).clip(lower=0)

planning["reorder_priority"] = np.where(
    planning["reorder_gap"] > 0,
    "Replenishment Review",
    "Stock Sufficient"
)

print("Planning records:", len(planning))
print("\nReplenishment review candidates:")
display(
    planning.sort_values(
        "reorder_gap", ascending=False
    ).head(15).round(2)
)

planning.to_csv(
    "../reports/inventory_recommendations.csv",
    index=False
)

print("\nSaved inventory_recommendations.csv")

Planning records: 500

Replenishment review candidates:


,Store ID,Product ID,Category,avg_daily_sales,avg_inventory,avg_forecast,target_stock_7_days,reorder_gap,reorder_priority
496,S005,P0020,Electronics,167.60,291.74,171.65,1201.52,909.78,Replenishment Review
472,S005,P0015,Furniture,164.62,294.89,170.97,1196.78,901.89,Replenishment Review
370,S004,P0015,Clothing,161.53,293.39,168.44,1179.09,885.70,Replenishment Review
284,S003,P0017,Toys,157.11,279.30,162.57,1137.98,858.69,Replenishment Review
25,S001,P0006,Clothing,156.80,290.84,162.70,1138.92,848.09,Replenishment Review
405,S005,P0002,Clothing,155.56,278.01,160.28,1121.98,843.97,Replenishment Review
393,S004,P0019,Groceries,155.82,287.19,161.52,1130.64,843.45,Replenishment Review
244,S003,P0009,Toys,154.94,278.02,159.98,1119.87,841.85,Replenishment Review
301,S004,P0001,Electronics,155.61,294.08,161.76,1132.30,838.22,Replenishment Review
340,S004,P0009,Clothing,153.20,281.67,159.23,1114.61,832.94,Replenishment Review



Saved inventory_recommendations.csv
